# Five-omics linear versus nonlinear age models

This analysis compares **Elastic Net** (linear) with a cubic-spline **GAM** and **LightGBM**, and produces the Layer-1 OOF predictions consumed by `02-aging_clock.py`.

Inputs are the already filtered and imputed `*_with_metadata.parquet` files in `02-papers/01-interim_data/omics_data`. No additional missingness/variance filtering or imputation is applied. Non-finite molecular values raise an error so they can be corrected upstream. The 40-70-year inclusion range is retained, and each omic uses its own full eligible cohort without requiring a five-omics intersection.

**Feature policy:** metabolomics, lipidomics, gut microbiome and oral microbiome use all supplied molecular features. Transcriptomics uses exactly **1,000** features per outer training fold (or all features if fewer are available). Only transcriptomics is ranked, using the existing Spearman/mutual-information candidate ranking and univariate cubic-spline GAM tests. The top 1,000 are selected by the smaller of the Spearman and GAM p-values; p-value cutoffs are annotations and do not reduce the requested feature count. All three models share the selected feature pool within each fold.

Participant-grouped 5-fold outer CV is repeated 10 times. Hyperparameters are tuned by participant-grouped 5-fold inner CV, and the nonlinear representative is chosen per outer fold using inner-CV MAE. The outer test fold is not used to choose between GAM and LightGBM or to rank transcriptomic features.

> Primary inference: paired outer-fold MAE, sklearn R2 and Pearson r, with one-sided Wilcoxon signed-rank tests, Cohen's dz and BH-FDR over five omics. Participant-cluster bootstrap intervals quantify improvement.

## Environment check

Uncomment the installation line only if the notebook kernel does not already contain LightGBM.

In [ ]:
# %pip install lightgbm
import sklearn
import lightgbm
print('scikit-learn:', sklearn.__version__)
print('lightgbm:', lightgbm.__version__)

## Run comparison

Training runs sequentially in this order: **metabolomics -> lipidomics -> gut microbiome -> oral microbiome -> transcriptomics**. Transcriptomics is trained last because it is the most expensive.

The default `publication` profile uses 5 outer folds x 10 repeats x 5 inner folds. The `quick` profile reduces this to 3 outer folds x 1 repeat x 3 inner folds, but keeps the same 1,000/all feature policy. Both profiles overwrite the Layer-1 output files, so rerun `publication` before reporting final results. The saved configuration records the source directory, training order, feature policy and input feature counts; each OOF row also records the number of features used.

In [ ]:
# Optional smoke test (3 outer folds × 1 repeat × 3 inner folds):
# %env AGING_CLOCK_PROFILE=quick

# Publication analysis (default):
%env AGING_CLOCK_PROFILE=publication
%run ./01-nonlinearity_model_comparison.py

## Results table

In [ ]:
display(metrics.round(3))
display(tests.round(4))
display(bootstrap.round(3))

## Figure

In [ ]:
from IPython.display import Image, display
display(Image(filename='figures/panel_b_linear_vs_nonlinear.png', width=1100))
display(Image(filename='figures/panel_b_mae_improvement_forest.png', width=800))
display(Image(filename='figures/extended_data_gam_lightgbm.png', width=1100))

## Reporting rule

The primary direction is `MAE_ElasticNet - MAE_nonlinear`; for sklearn $R^2$ and Pearson $r$, the contrast is `nonlinear - ElasticNet`, so positive values favor the nonlinear representative. Significance stars use BH-adjusted paired Wilcoxon p values across the five omics. `nonlinearity_inner_tuning.csv` records which nonlinear model won inside each outer training fold, while `nonlinearity_fold_metrics.csv` retains the complete GAM and LightGBM distributions for Extended Data. Bootstrap intervals resample participants (not rows), retaining repeated observations within participant.

A predictive advantage supports nonlinear predictive structure but is not, by itself, a mechanistic proof of biological nonlinearity.